# Stage 0 — Cross-Silo Label Harmonization

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PROJECT = "/content/drive/MyDrive/Depression_FL_Project"

# EEG / MODMA: PHQ-9 lives in the subjects-information Excel
EEG_META_XLSX = f"{PROJECT}/data/raw/MODMA/subjects_information_EEG_3channels_resting_lanzhou_2015.xlsx"

AUDIO_ROOT = f"{PROJECT}/data/raw/DAIC_WOZ"
TRAIN_CSV  = f"{AUDIO_ROOT}/train_split_Depression_AVEC2017.csv"
DEV_CSV    = f"{AUDIO_ROOT}/dev_split_Depression_AVEC2017.csv"
TEST_CSV   = f"{AUDIO_ROOT}/full_test_split.csv"

TEXT_PATIENTS = f"{PROJECT}/output/PDCH/processed_v1/pdch_patients.parquet"

OUT = f"{PROJECT}/fl/labels_v1"
import os, glob, json
import numpy as np, pandas as pd
os.makedirs(OUT, exist_ok=True)

if not os.path.exists(TEXT_PATIENTS):
    hits = (glob.glob(f"{PROJECT}/**/pdch_patients.parquet", recursive=True)
            + glob.glob("/content/drive/MyDrive/**/pdch_patients.parquet", recursive=True))
    if hits:
        TEXT_PATIENTS = hits[0]; print("Found text table at:", TEXT_PATIENTS)

for p in (EEG_META_XLSX, TRAIN_CSV, DEV_CSV, TEST_CSV, TEXT_PATIENTS):
    print(f"  exists={os.path.exists(p)}  {p}")

In [ ]:
PHQ_BANDS    = [(-1, 4, 0, "none_minimal"), (4, 14, 1, "mild_moderate"), (14, 99, 2, "severe")]
HAMD17_BANDS = [(-1, 7, 0, "none_minimal"), (7, 18, 1, "mild_moderate"), (18, 99, 2, "severe")]
UNIFIED_NAMES = {0: "none_minimal", 1: "mild_moderate", 2: "severe"}

def _band(score, bands):
    if score is None or (isinstance(score, float) and np.isnan(score)): return (np.nan, None)
    for lo, hi, code, name in bands:
        if lo < score <= hi: return (code, name)
    return (np.nan, None)

def map_scores(scores, instrument):
    bands = PHQ_BANDS if instrument.lower().startswith("phq") else HAMD17_BANDS
    return np.array([_band(float(s), bands)[0] if s is not None and not pd.isna(s) else np.nan
                     for s in scores], dtype=float)

def build_silo_frame(silo, subject_ids, raw_scores, instrument, split=None):
    y = map_scores(raw_scores, instrument)
    return pd.DataFrame({"silo": silo, "subject_id": [str(s) for s in subject_ids],
        "instrument": instrument.lower(), "raw_score": np.asarray(raw_scores, dtype=float),
        "y_unified": y,
        "y_name": [UNIFIED_NAMES.get(int(c), None) if not pd.isna(c) else None for c in y],
        "split": (split if split is not None else "all")})

assert [int(map_scores([s],"phq8")[0]) for s in (4,5,14,15)] == [0,1,1,2]
assert [int(map_scores([s],"hamd17")[0]) for s in (7,8,18,19)] == [0,1,1,2]
print("policy boundary check passed")

In [ ]:
meta = pd.read_excel(EEG_META_XLSX, sheet_name="Sheet1").iloc[:, :11]
meta.columns = ["subject_id","type","age","gender","education_years",
                "PHQ9","CTQ_SF","LES","SSRS","GAD7","PSQI"]
meta = meta.dropna(subset=["subject_id"]).copy()
meta["subject_id"] = meta["subject_id"].astype(str).str.strip()
meta["PHQ9"] = pd.to_numeric(meta["PHQ9"], errors="coerce")

eeg_frame = build_silo_frame("eeg", meta["subject_id"], meta["PHQ9"], "phq9")
if "type" in meta.columns:
    eeg_frame["clinical_type"] = meta["type"].astype(str).str.upper().values
print("EEG subjects:", len(eeg_frame), "| PHQ-9 range:",
      np.nanmin(meta.PHQ9), "-", np.nanmax(meta.PHQ9))
display(eeg_frame.groupby("y_name").size())

In [ ]:
def _read_daic(path, split):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    idcol = next(c for c in df.columns if c.lower() == "participant_id")
    scol  = "PHQ8_Score" if "PHQ8_Score" in df.columns else "PHQ_Score"
    out = df[[idcol, scol]].copy(); out.columns = ["pid", "phq8"]
    out["pid"] = out["pid"].astype(int); out["phq8"] = out["phq8"].astype(float)
    out["split"] = split; return out

daic = pd.concat([_read_daic(TRAIN_CSV,"train"), _read_daic(DEV_CSV,"dev"),
                  _read_daic(TEST_CSV,"test")], ignore_index=True)
EXCLUDED = {342, 394, 398, 460}
daic = daic[~daic.pid.isin(EXCLUDED)].reset_index(drop=True)
audio_frame = build_silo_frame("audio", daic["pid"], daic["phq8"], "phq8", split=daic["split"].values)
print("Audio subjects:", len(audio_frame))
display(audio_frame.groupby(["split","y_name"]).size().unstack(fill_value=0))

In [ ]:
pat = pd.read_parquet(TEXT_PATIENTS); pat["patient_id"] = pat["patient_id"].astype(str)
text_frame = build_silo_frame("text", pat["patient_id"], pat["hamd_total"], "hamd17")

if "sev3_code" in pat.columns:
    old = pat["sev3_code"].to_numpy(dtype=float); new = text_frame["y_unified"].to_numpy(dtype=float)
    flipped = int(np.nansum(old != new))
    print(f"TEXT label flips vs locked sev3: {flipped} / {len(pat)} patients")
    if flipped:
        chg = pat.loc[old != new, ["patient_id","hamd_total"]].copy()
        chg["old"] = old[old != new]; chg["new"] = new[old != new]; display(chg)
        print("-> re-evaluate the locked text probe on y_unified (cheap).")
print("Text subjects:", len(text_frame)); display(text_frame.groupby("y_name").size())

In [ ]:
def harmonization_report(frames):
    all_df = pd.concat(frames, ignore_index=True); rows, warns = [], []
    for silo, g in all_df.groupby("silo"):
        c = g["y_unified"].value_counts(dropna=True).to_dict()
        rows.append({"silo": silo, "instrument": g.instrument.iloc[0], "n_subjects": len(g),
                     "none_minimal": int(c.get(0.0,0)), "mild_moderate": int(c.get(1.0,0)),
                     "severe": int(c.get(2.0,0)), "n_missing_label": int(g.y_unified.isna().sum())})
        for code,name in UNIFIED_NAMES.items():
            k = int(c.get(float(code),0))
            if k == 0:
                warns.append(f"[EMPTY] silo='{silo}' class {code} ({name}): 0 subjects "
                             f"-> no prototype; must be supplied by cross-silo transfer.")
            elif k < 5:
                warns.append(f"[TINY]  silo='{silo}' class {code} ({name}): {k} subjects "
                             f"-> high-variance prototype; seed-average + report CIs.")
    rep = (pd.DataFrame(rows).set_index("silo")
           [["instrument","n_subjects","none_minimal","mild_moderate","severe","n_missing_label"]])
    return rep, warns

frames = [eeg_frame, audio_frame, text_frame]
report, warnings = harmonization_report(frames)
print("="*70 + "\nUNIFIED 3-CLASS DISTRIBUTION PER SILO\n" + "="*70)
display(report)
print("\nWARNINGS (read these -- they shape the FL design):")
for w in warnings: print("  ", w)
if not warnings: print("   none -- all silos populate all three classes.")

In [ ]:
# ---------- Integrity checks ----------
harm = pd.concat(frames, ignore_index=True)
checks = {
 "unique_id_within_silo": bool(harm.groupby("silo").subject_id.apply(lambda s: s.is_unique).all()),
 "labels_in_012_or_nan": bool(harm.y_unified.dropna().isin([0,1,2]).all()),
 "no_silo_fully_unlabeled": bool(harm.groupby("silo").y_unified.apply(lambda s: s.notna().any()).all()),
 "raw_scores_nonneg": bool((harm.raw_score.dropna() >= 0).all()),
}
for k,v in checks.items(): print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "integrity check failed -- inspect above"
print("\nAll integrity checks passed.")

In [ ]:
harm_out = f"{OUT}/fl_harmonized_labels.csv"
harm.to_csv(harm_out, index=False)
summary = {"policy": {"PHQ_BANDS": PHQ_BANDS, "HAMD17_BANDS": HAMD17_BANDS, "unified": UNIFIED_NAMES},
  "per_silo": json.loads(report.reset_index().to_json(orient="records")),
  "warnings": warnings, "n_total": int(len(harm))}
json.dump(summary, open(f"{OUT}/fl_harmonization_summary.json","w"), indent=2, ensure_ascii=False)
print("saved:\n ", harm_out, "\n ", f"{OUT}/fl_harmonization_summary.json")